## About Dataset
Context
This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content
5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset.
Columns

- asin - ID of the product, like B000FA64PK
- helpful - helpfulness rating of the review - example: 2/3.
- overall - rating of the product.
- reviewText - text of the review (heading).
- reviewTime - time of the review (raw).
- reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
- reviewerName - name of the reviewer.
- summary - summary of the review (description).
- unixReviewTime - unix timestamp.

Acknowledgements
This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

Inspiration
- Sentiment analysis on reviews.
- Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.
- Fake reviews/ outliers.
- Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
- Any other interesting analysis
#### Best Practises
1. Preprocessing And Cleaning
2. Train Test Split
3. BOW,TFIDF,Word2vec
4. Train ML algorithms

In [1]:
# Load the dataset
import pandas as pd
data=pd.read_csv('all_kindle_review.csv')
data.head()
df=data[['reviewText','rating']]
df.head()
df.shape

(12000, 2)

In [2]:
## Missing Values
df.isnull().sum()
df['rating'].unique()
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [3]:
## Preprocessing And Cleaning
## postive review is 1 and negative review is 0
df['rating']=df['rating'].apply(lambda x:0 if x<3 else 1)
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [4]:
## 1. Lower All the cases
df['reviewText']=df['reviewText'].str.lower()
df.head()
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     c:\DS2026\Python\.venv\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [5]:
!pip install beautifulsoup4

In [8]:
import re
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

special_char_pattern = re.compile(r'[^a-zA-Z0-9\s-]+')
url_pattern = re.compile(r'https?://\S+|www\.\S+')

def preprocess_text(text):
    text = str(text)

    text = url_pattern.sub('', text)
    text = special_char_pattern.sub('', text)

    text = " ".join(
        word for word in text.split()
        if word.lower() not in stop_words
    )

    return text

df['reviewText'] = df['reviewText'].apply(preprocess_text)

In [10]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [11]:
## Lemmatizer
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

df['reviewText']=df['reviewText'].apply(lambda x:lemmatize_words(x))
df.head()


,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [12]:
## Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['reviewText'],df['rating'],
                                              test_size=0.20)
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
X_train_bow=bow.fit_transform(X_train).toarray()
X_test_bow=bow.transform(X_test).toarray()
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train).toarray()
X_test_tfidf=tfidf.transform(X_test).toarray()
X_train_bow
from sklearn.naive_bayes import GaussianNB
nb_model_bow=GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf=GaussianNB().fit(X_train_tfidf,y_train)
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
y_pred_bow=nb_model_bow.predict(X_test_bow)
y_pred_tfidf=nb_model_bow.predict(X_test_tfidf)
confusion_matrix(y_test,y_pred_bow)
print("BOW accuracy: ",accuracy_score(y_test,y_pred_bow))
confusion_matrix(y_test,y_pred_tfidf)
print("TFIDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))


BOW accuracy:  0.5670833333333334
TFIDF accuracy:  0.5683333333333334


In [13]:
# Step 6: Function to predict sentiment for new reviews
def predict_review(review_text, model, vectorizer, method="BOW"):
    # Clean the text using the same preprocessing pipeline
    cleaned = clean_text(review_text)
    # Transform using the chosen vectorizer
    features = vectorizer.transform([cleaned]).toarray()
    # Predict sentiment
    prediction = model.predict(features)[0]
    sentiment = "Positive" if prediction == 1 else "Negative"
    print(f"Original Review: {review_text}")
    print(f"Cleaned Review: {cleaned}")
    print(f"Prediction ({method}): {sentiment}")
    return sentiment


In [14]:
# Example 1: Positive review
predict_review("This book was absolutely amazing, I loved every chapter!", nb_model_bow, bow, "BOW")
predict_review("This book was absolutely amazing, I loved every chapter!", nb_model_tfidf, tfidf, "TFIDF")

# Example 2: Negative review
predict_review("Terrible writing, waste of money. Do not buy.", nb_model_bow, bow, "BOW")
predict_review("Terrible writing, waste of money. Do not buy.", nb_model_tfidf, tfidf, "TFIDF")

# Example 3: Neutral/critical review
predict_review("The story was good but the formatting was awful.", nb_model_bow, bow, "BOW")
predict_review("The story was good but the formatting was awful.", nb_model_tfidf, tfidf, "TFIDF")


NameError: name 'clean_text' is not defined